In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.metrics import classification_report

In [2]:
ckd_dataset = pd.read_csv("PreprocessedKidneyDisease.csv")
ckd_dataset

,id,age,bp,sg,al,su,bgr,bu,sc,sod,...,pc,pcc,ba,htn,dm,cad,appet,pe,ane,classification
0,0.0,48.0,80.0,1.020,1.0,0.0,121.000000,36.0,1.2,137.528754,...,1,0,0,1,1,0,0,0,0,0
1,1.0,7.0,50.0,1.020,4.0,0.0,148.036517,18.0,0.8,137.528754,...,1,0,0,0,0,0,0,0,0,0
2,2.0,62.0,80.0,1.010,2.0,3.0,423.000000,53.0,1.8,137.528754,...,1,0,0,0,1,0,1,0,1,0
3,3.0,48.0,70.0,1.005,4.0,0.0,117.000000,56.0,3.8,111.000000,...,0,1,0,1,0,0,1,1,1,0
4,4.0,51.0,80.0,1.010,2.0,0.0,106.000000,26.0,1.4,137.528754,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,395.0,55.0,80.0,1.020,0.0,0.0,140.000000,49.0,0.5,150.000000,...,1,0,0,0,0,0,0,0,0,1
396,396.0,42.0,70.0,1.025,0.0,0.0,75.000000,31.0,1.2,141.000000,...,1,0,0,0,0,0,0,0,0,1
397,397.0,12.0,80.0,1.020,0.0,0.0,100.000000,26.0,0.6,137.000000,...,1,0,0,0,0,0,0,0,0,1
398,398.0,17.0,60.0,1.025,0.0,0.0,114.000000,50.0,1.0,135.000000,...,1,0,0,0,0,0,0,0,0,1


In [3]:
dataset=ckd_dataset

In [4]:
indep_x = dataset.drop(["classification", "id"], axis=1)
dep_y = dataset["classification"]


In [5]:
def selectKBest(indep_x, dep_y, n):
    test = SelectKBest(chi2, k=n)
    fit1 = test.fit(indep_x, dep_y)
    selectk_features = fit1.transform(indep_x)
    selected_columns = indep_x.columns[fit1.get_support()]
    return selectk_features, selected_columns

def split_scalar(indep_x, dep_y):
    x_train, x_test, y_train, y_test = train_test_split(indep_x, dep_y, test_size = 0.25, random_state=0)
    sc = StandardScaler()
    x_train = sc.fit_transform(x_train)
    x_test = sc.transform(x_test)
    return x_train, x_test, y_train, y_test


In [9]:
Kbest, ColBest = selectKBest(indep_x, dep_y, 8)
x_train, x_test, y_train, y_test = split_scalar(Kbest, dep_y)


In [7]:
def cm_prediction(Classifier, x_test, y_test):
    from sklearn.metrics import confusion_matrix
    from sklearn.metrics import accuracy_score
    from sklearn.metrics import classification_report
    y_pred = Classifier.predict(x_test)
    cm = confusion_matrix(y_test, y_pred)
    Accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    return Classifier, Accuracy, report, x_test, y_test, cm


In [8]:
def random(x_train, y_train, x_test, y_test):
    from sklearn.ensemble import RandomForestClassifier
    Classifier = RandomForestClassifier(n_estimators =10, criterion = 'entropy', random_state=0)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier,x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm

In [26]:
Classifier, Accuracy, report, x_test, y_test, cm = random(x_train, y_train, x_test, y_test)
print(report)
#print(cm)
print(f"Accuracy of the Random Forest Model for k=8 and CKD preprocessed dataset is : {Accuracy:.2%}")
print("\nBest Eight columns chosen using SelectKBest:")
for i, column in enumerate(ColBest, start=1):
    print(f"{i}.{column}")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        62
           1       1.00      1.00      1.00        38

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100

Accuracy of the Random Forest Model for k=8 and CKD preprocessed dataset is : 100.00%

Best Eight columns chosen using SelectKBest:
1.age
2.al
3.bgr
4.bu
5.sc
6.hemo
7.pcv
8.wc


In [27]:
import pickle

In [28]:
filename = "finalized_model_RandomForest.sav"

In [29]:
pickle.dump(Classifier, open(filename,'wb'))